# 03 — Live experiment against the deployed MSK cluster

Unlike `00`-`02`, which point at `$FDAI_TARGET` (local by default), this one is
pinned to the real cloud cluster on purpose — there is no ambiguity about which
broker you are looking at. A few small, quick experiments on whatever is
actually flowing right now, not a full analysis (see `01_explore_trades.ipynb`
for that).

**Prerequisite:** launch Jupyter with `make notebook TARGET=msk`. The launch
preflight handles the AWS profile, SSO refresh, current operator IP, Terraform
reconciliation, and Kafka metadata check before this notebook opens.

In [7]:
import devlab
from devlab import frames

target = devlab.from_terraform()  # the deployed MSK cluster, explicitly — not $FDAI_TARGET
target

Target(name='msk', bootstrap='b-1-public.fdaikafka.ru8ywo.c23.kafka.us-east-1.amazonaws.com:9196,b-2-public.fdaikafka.ru8ywo.c23.kafka.us-east-1.amazonaws.com:9196', sasl_username='fdai-producer')

## Is it actually live right now

Cheap, and worth doing before anything else: reads from `latest` for 10
seconds. A zero result means no matching trade arrived during that short
window; it is not by itself evidence that the producer is down.

In [8]:
report = devlab.rate(target, seconds=10.0)
print(f"{report.messages} trades in {report.seconds:.1f}s = {report.per_second:.1f}/s")
report.by_venue

0 trades in 10.3s = 0.0/s


{}

## Grab a batch of real trades

Bounded by both a count and a clock, so this returns even against a quiet
topic. `offset_reset="earliest"` reads what's already retained rather than
only what arrives from this point on.

In [ ]:
records = devlab.collect(target, limit=2_000, seconds=30.0, offset_reset="earliest")
df = frames.trades_frame(records)
print(f"{len(df):,} trades  {df['event_ts'].min()} .. {df['event_ts'].max()}")
df.head()

## Experiment 1 — who's trading what

Per venue and instrument: trade count, volume, and notional. Sorted by
notional, so whatever's actually moving money floats to the top.

In [ ]:
(
    df.groupby(["venue", "instrument_id"], observed=True)
    .agg(trades=("trade_id", "count"), volume=("size", "sum"), notional=("notional", "sum"))
    .sort_values("notional", ascending=False)
)

## Experiment 2 — price, live

Whichever instrument traded the most in this window, plotted as-is (no
resampling) — the rawest possible look at the tape.

In [ ]:
top_instrument = df["instrument_id"].value_counts().idxmax()
subset = df[df["instrument_id"] == top_instrument]
axis = subset.plot(x="event_ts", y="price", figsize=(10, 3), title=f"{top_instrument} — live")
axis.set_xlabel("event time (UTC)")

## Experiment 3 — a quick VWAP

Dedupe first (a repaired trade can appear twice — see `02_prototype_silver.ipynb`
for why), then 1-minute bars. Just the tail, since this is meant to be a quick
look, not the full analysis.

In [ ]:
bars = frames.bars(frames.dedupe(df), freq="1min")
bars.tail()